### Option 1: TED talks via Huggingface

#### Load Dataset

In [ ]:
from datasets import load_dataset
import os
import re
import numpy as np
import soundfile as sf
from collections import defaultdict

tedlium_dataset = load_dataset(
    "LIUM/tedlium",
    name="release1",
    split="test",
    streaming=True
)

#### Get n samples

In [32]:
def clean_transcript(text):
    if "ignore_time_segment_in_scoring" in text:
        return ""
    return re.sub(r"\b(\w)\s+'(\w)", r"\1\2", text).strip()

def save_n_speakers_by_duration(dataset, n=10, duration_sec=120, out_folder="data"):
    os.makedirs(out_folder, exist_ok=True)
    speaker_data = defaultdict(lambda: {"audio": [], "text": [], "sr": None})

    for sample in dataset:
        speaker = sample.get("speaker_id")
        if not speaker:
            continue
        text = clean_transcript(sample["text"])
        if not text:
            continue
        audio = sample["audio"]["array"]
        sr = sample["audio"]["sampling_rate"]
        speaker_data[speaker]["audio"].append(audio)
        speaker_data[speaker]["text"].append(text)
        speaker_data[speaker]["sr"] = sr

    count = 0
    for speaker, data in speaker_data.items():
        sr = data["sr"]
        needed_samples = duration_sec * sr
        collected_audio, collected_text, collected = [], [], 0

        for audio, text in zip(data["audio"], data["text"]):
            if collected >= needed_samples:
                break
            take = min(len(audio), needed_samples - collected)
            collected_audio.append(audio[:take])
            frac = take / len(audio)
            if frac > 0.5:
                words = text.split()
                collected_text.append(" ".join(words[:int(len(words) * frac)]))
            collected += take

        if collected < needed_samples:
            continue

        out_audio = np.concatenate(collected_audio)
        out_text = " ".join(collected_text)
        count += 1
        sf.write(os.path.join(out_folder, f"ted_sample_{count}_speaker_{speaker}.wav"), out_audio, sr)
        with open(os.path.join(out_folder, f"ted_sample_{count}_speaker_{speaker}.txt"), "w", encoding="utf-8") as f:
            f.write(out_text)
        print(f"Saved sample {count} for speaker {speaker}")

        if count >= n:
            break

save_n_speakers_by_duration(tedlium_dataset, n=10, duration_sec=120, out_folder="data")

Saved sample 1 for speaker AimeeMullins
Saved sample 2 for speaker BillGates
Saved sample 3 for speaker DanBarber
Saved sample 4 for speaker DanielKahneman
Saved sample 5 for speaker EricMead_2009P_EricMead
Saved sample 6 for speaker GaryFlake
Saved sample 7 for speaker JamesCameron
Saved sample 8 for speaker JaneMcGonigal
Saved sample 9 for speaker MichaelSpecter
Saved sample 10 for speaker RobertGupta


### Option 2: Audio books

Download 'train-clean-100.tar.gz [6.3G]' from LibriSpeech ASR corpus

 [Get gz file here](https://github.com/josephnguyen0413/02467_Assignment2)

### Extracting downdloaded gz file

In [16]:
import tarfile

# Open and extract the .tar.gz file
with tarfile.open('TEDLIUM_release1.tar.gz', 'r:gz') as tar:
    tar.extractall(path='TED_talks')  # you can set your target directory here


### Create audio file

In [ ]:
import os
import soundfile as sf
import numpy as np

# Set your input and output paths
input_folder = "audio_files/LibriSpeech/train-clean-100/32/21625" # 32 = reader id, 21625 = audio book id
output_file = "brownie_beaver_4_min.wav"

# Find and sort all .flac files in the folder
flac_files = sorted([
    f for f in os.listdir(input_folder)
    if f.lower().endswith(".flac")
])

if not flac_files:
    raise FileNotFoundError(f"No .flac files found in: {input_folder}")

# Combine audio
combined_audio = []
sample_rate = None

for flac_file in flac_files:
    path = os.path.join(input_folder, flac_file)
    audio_data, sr = sf.read(path)
    
    # Make sure sample rate matches
    if sample_rate is None:
        sample_rate = sr
    elif sr != sample_rate:
        raise ValueError(f"Sample rate mismatch in {flac_file}: {sr} != {sample_rate}")
    
    combined_audio.append(audio_data)

# Concatenate all audio data
final_audio = np.concatenate(combined_audio)

# Save as a .wav file
sf.write(output_file, final_audio, sample_rate)
print(f"Combined audio saved to: {output_file}")


Combined audio saved to: brownie_beaver_4_min.wav


### Getting transcipt for audio file

In [2]:
import os

# Paths
input_folder = "audio_files/LibriSpeech/train-clean-100/32/21625"
transcript_path = os.path.join(input_folder, "32-21625.trans.txt")
output_transcript = "brownie_beaver_4_min.txt"

# Load and parse the .trans file
transcript_map = {}

with open(transcript_path, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        parts = line.strip().split(" ", 1)
        if len(parts) == 2:
            key, text = parts
            transcript_map[key] = text

# Sort the keys to match audio order
sorted_keys = sorted(transcript_map.keys())

# Combine the transcriptions into one continuous line
combined_text = " ".join([transcript_map[k] for k in sorted_keys])

# Save to a text file
with open(output_transcript, "w", encoding="utf-8") as out_f:
    out_f.write(combined_text)

print(f" Combined transcript saved to: {output_transcript}")


 Combined transcript saved to: brownie_beaver_4_min.txt
